# 01 — Treino: LIBERO com fusão de linguagem FiLM (Fase 3)

Notebook fino: toda a lógica vive em `src/act_lang/`. Aqui só orquestração e visualização.

Configurado para `configs/libero_object_language.py::CONFIG_FILM` — 10 tarefas do
`libero_object`, mecanismo de fusão FiLM. Para rodar outra fase/mecanismo, troque
só a linha de import na célula de dados (comentário nela mostra as alternativas).

In [ ]:
# --- Setup: detecta Colab vs ambiente local, e configura cada um ---
# Precisa ser um check MÍNIMO, sem depender de act_lang -- no Colab é esta
# célula que instala o pacote; ainda não existe antes disso rodar.
try:
    import google.colab
    _in_colab = True
except ImportError:
    _in_colab = False

import sys
from pathlib import Path

if _in_colab:
    # Repo privado? Guarde um token no Secrets do Colab e use:
    #   from google.colab import userdata; token = userdata.get("GH_TOKEN")
    #   !git clone https://{token}@github.com/rafaelheydt/act-lang.git /content/act-lang
    #
    # %cd /content ANTES do rm -rf: se uma execução anterior desta célula
    # deixou o shell dentro de /content/act-lang, apagar essa pasta com o
    # shell "sentado" nela quebra o cwd do processo (erros "getcwd: cannot
    # access parent directories") e derruba até o git clone seguinte.
    %cd /content
    !rm -rf /content/act-lang
    !git clone https://github.com/rafaelheydt/act-lang.git /content/act-lang
    %cd /content/act-lang
    # extra "[language]" traz o sentence-transformers usado pelos 3
    # mecanismos de fusão (Fase 3, ex: FiLM) -- sem ele, build_fusion()
    # constrói o objeto normalmente, mas o 1o encode_text() real falha ao
    # tentar importar sentence_transformers.
    !pip install -q -e ".[language]" "lerobot[libero]"

    # "configs/" fica FORA de src/ de propósito (configs de experimento
    # editáveis sem reinstalar nada) -- por isso não é abrangido pelo pip
    # install -e . Alguns kernels IPython não resolvem import a partir do
    # cwd dinamicamente, então o insert explícito é a forma confiável.
    if "/content/act-lang" not in sys.path:
        sys.path.insert(0, "/content/act-lang")
else:
    # Local (ex: quando a sessão do Colab expira): pré-requisito é rodar
    # UMA VEZ no terminal, dentro do seu ambiente conda, fora do notebook:
    #   git clone https://github.com/rafaelheydt/act-lang.git
    #   cd act-lang && pip install -e ".[language]" "lerobot[libero]"
    # Sem a dança do restart do Colab -- pip install -e . nesse mesmo
    # processo já funciona, mas rodar ele DENTRO do notebook seria menos
    # confiável (mistura ambientes); melhor deixar isso no terminal.
    print("Ambiente local detectado.")
    try:
        import act_lang
        repo_root_local = Path(act_lang.__file__).resolve().parents[2]
        if str(repo_root_local) not in sys.path:
            sys.path.insert(0, str(repo_root_local))  # pro "from configs...."
        print(f"act_lang encontrado em: {repo_root_local}")
    except ImportError:
        raise RuntimeError(
            "act_lang não instalado neste ambiente. No terminal:\n"
            "  git clone https://github.com/rafaelheydt/act-lang.git\n"
            '  cd act-lang && pip install -e ".[language]" "lerobot[libero]"\n'
            "e reinicie o kernel deste notebook."
        )

import os
os.environ.setdefault("MUJOCO_GL", "egl")  # headless (Colab); local com X11 também funciona

from act_lang.utils.runtime import is_colab, pick_device, describe_devices, get_checkpoint_dir

# SEM autoreload no Colab: o IPython pré-instalado lá (pinado em 7.34.0
# pelo pacote google-colab) quebra com ele no runtime atual, e forçar
# upgrade do IPython quebra drive.mount()/exibição de vídeo em troca.
# Local, se quiser, ative manualmente:
#   %load_ext autoreload
#   %autoreload 2
#
# IMPORTANTE (só Colab): depois desta célula rodar pela 1a vez em CADA
# runtime novo, faça Runtime > Restart session antes de continuar -- o
# Python só lê o registro do "pip install -e ." (.pth) na inicialização
# do interpretador. Sem o restart, "import act_lang" falha mesmo com tudo
# instalado corretamente. (Local: não é necessário, pip install -e . já
# foi feito antes de abrir este notebook.)

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from configs.libero_object_language import CONFIG_FILM as cfg
# Outras fases/mecanismos -- troque a linha acima por:
#   from configs.libero_single_task import CONFIG as cfg              # Fase 1
#   from configs.libero_object_multitask import CONFIG as cfg         # Fase 2
#   from configs.libero_object_language import CONFIG_TOKEN as cfg    # Fase 3, token
#   from configs.libero_object_language import CONFIG_CROSS_ATTN as cfg  # Fase 3, cross-attn
from act_lang.data.libero import (
    REPO_ID, LiberoActBridge, filter_episodes_by_tasks, get_episode_task_labels,
    make_delta_timestamps, split_episodes_min_holdout, split_episodes_stratified,
)
from act_lang.data.normalize import MinMaxNormalizer

print(describe_devices())
device = pick_device(preferred_index=cfg.get("device_index"))  # None = auto (mais memória livre)
print(f"\nusando: {device}")

In [ ]:
# --- Dados: filtro por tarefa(s), split por episódio, loaders ---
meta = LeRobotDatasetMetadata(REPO_ID)
full_dataset = LeRobotDataset(REPO_ID)

episode_ids = filter_episodes_by_tasks(meta, full_dataset, cfg["task_texts"])
episode_task_labels = get_episode_task_labels(meta, full_dataset, episode_ids)

# split por tarefa -- "fraction" (default, comportamento da Fase 1) reserva
# uma % por tarefa; "min_holdout" (Fase 2) reserva um número FIXO pequeno
# (ex: 1) por tarefa, comparável entre tarefas mesmo com poucos episódios.
# Com 1 tarefa só, os dois dão o mesmo resultado do split simples.
val_strategy = cfg.get("val_strategy", "fraction")
if val_strategy == "min_holdout":
    train_ids, val_ids = split_episodes_min_holdout(
        episode_ids, episode_task_labels, cfg.get("n_val_per_task", 1), cfg["seed"]
    )
else:
    train_ids, val_ids = split_episodes_stratified(
        episode_ids, episode_task_labels, cfg["val_frac"], cfg["seed"]
    )
print(f"episódios: {len(episode_ids)} -> train {len(train_ids)} | val {len(val_ids)} (val_strategy={val_strategy!r})")

if len(cfg["task_texts"]) > 1:
    from collections import Counter
    train_por_tarefa = Counter(episode_task_labels[e] for e in train_ids)
    val_por_tarefa = Counter(episode_task_labels[e] for e in val_ids)
    print("\nepisódios por tarefa (train | val):")
    for tarefa in sorted(cfg["task_texts"]):
        print(f"  {train_por_tarefa.get(tarefa, 0):>2} | {val_por_tarefa.get(tarefa, 0):>2}  {tarefa}")
# ATENÇÃO: com poucas dezenas de episódios, val é pequeno — métricas de
# validação são ruidosas; interprete o "melhor checkpoint" com essa lente.

delta_ts = make_delta_timestamps(meta.fps, cfg["obs_horizon"], cfg["pred_horizon"])
train_dataset = LeRobotDataset(REPO_ID, episodes=train_ids, delta_timestamps=delta_ts)
val_dataset = LeRobotDataset(REPO_ID, episodes=val_ids, delta_timestamps=delta_ts)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=cfg["batch_size"], shuffle=True,
    num_workers=0, drop_last=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0,
)

state_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "observation.state").to(device)
action_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "action").to(device)
bridge = LiberoActBridge(state_norm, action_norm)

In [ ]:
# --- Modelo + optimizer ---
from act_lang.models.act import ACT
from act_lang.models.backbone import freeze_batchnorm
from act_lang.models.fusion import build_fusion
from act_lang.training.optim import build_optimizer

# cfg.get: Fases 1/2 não têm "fusion_type" -> None -> baseline sem linguagem
fusion = build_fusion(cfg.get("fusion_type"), d_model=cfg["d_model"])
if fusion is not None:
    print(f"fusão de linguagem: {cfg['fusion_type']}")

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"],
    d_model=cfg["d_model"], latent_dim=cfg["latent_dim"],
    chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], pretrained_backbone=True,
    decoder_style=cfg["decoder_style"], fusion=fusion,
)
if cfg["freeze_bn"]:
    freeze_batchnorm(model.vision_backbone)
model = model.to(device)
print(f"parâmetros: {sum(p.numel() for p in model.parameters()):,}")

optimizer = build_optimizer(model, cfg["lr"], cfg["lr_backbone"], cfg["weight_decay"])

In [ ]:
# --- Smoke test: um batch real, forward + backward ---
from act_lang.training.loss import act_loss

batch = next(iter(train_loader))
images, state, actions, is_pad, task_texts = bridge(batch, device)
print(f"images {tuple(images.shape)} | state {tuple(state.shape)} | actions {tuple(actions.shape)}")

pred, mu, logvar = model(images, state, actions=actions, is_pad=is_pad, task_texts=task_texts)
loss, recon, kld = act_loss(pred, actions, mu, logvar, is_pad, cfg["kl_weight"], cfg["free_bits"])
loss.backward(); optimizer.zero_grad()
print(f"loss {loss.item():.4f} | recon {recon.item():.4f} | kld {kld.item():.4f}")

In [ ]:
# --- Checkpoint dir (Drive no Colab / pasta local fora dele) + treino ---
checkpoint_dir = get_checkpoint_dir(cfg["experiment_name"])
print(f"checkpoints em: {checkpoint_dir}")

from act_lang.training.loop import fit

# retomar de sessão caída:
#   from act_lang.training.checkpoints import load_checkpoint
#   start_epoch, history = load_checkpoint(
#       checkpoint_dir / "last_checkpoint.pt", model, optimizer, device)
#   e passe start_epoch=start_epoch, history=history ao fit(...)

history = fit(
    model, train_loader, val_loader, bridge, optimizer, device,
    checkpoint_dir=checkpoint_dir, num_epochs=cfg["num_epochs"],
    kl_weight=cfg["kl_weight"], free_bits=cfg["free_bits"],
    grad_clip_norm=cfg["grad_clip_norm"], patience=cfg["patience"],
    checkpoint_every=cfg["checkpoint_every"],
)

In [ ]:
# --- Curvas ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss total (recon + kl_weight*KL)")

axes[1].plot(history["train_recon"], label="train (z~q)")
axes[1].plot(history["val_recon"], label="val (z=mu)")
axes[1].plot(history["val_recon_z0"], label="val (z=0)", linestyle="--")
axes[1].set_title("Recon L1 — z0 é a métrica de seleção")

axes[2].plot(history["train_kld"], label="train")
axes[2].plot(history["val_kld"], label="val")
axes[2].set_title("kld_raw")

axes[3].plot(history["train_mu_abs_mean"], label="train")
axes[3].plot(history["val_mu_abs_mean"], label="val")
axes[3].axhline(0, color="gray", linestyle=":", linewidth=1)
axes[3].set_title("|mu| médio — perto de 0 = posterior colapsado")

for ax in axes:
    ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.show()